# 06 — Model Quantization for Edge Deployment

## Goal

Quantization reduces the numerical precision used by a model. This can reduce model size and may improve inference speed on supported hardware, with a possible accuracy trade-off.

We will convert the small CNN to TensorFlow Lite (LiteRT) format and compare:
- Float32 baseline.
- Dynamic-range quantization.
- Full integer INT8 quantization.

Run Notebook 03 first. Conversion APIs can vary by TensorFlow version; if conversion fails, preserve the full traceback rather than changing package versions blindly.


In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import cifar10

os.makedirs("../artifacts", exist_ok=True)
model = keras.models.load_model("../artifacts/cifar10_cnn.keras")

(_, _), (x_test_raw, y_test_raw) = cifar10.load_data()
x_test = x_test_raw.astype("float32") / 255.0
y_test = y_test_raw.squeeze()

print("Model input:", model.input_shape)
print("Test set:", x_test.shape)


## 1. Convert the float32 model

Save a baseline TFLite model first. We will compare the exported file sizes.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
float32_bytes = converter.convert()

float32_path = "../artifacts/cifar10_float32.tflite"
with open(float32_path, "wb") as f:
    f.write(float32_bytes)

print("Saved:", float32_path)
print("Float32 model size (MiB):", round(len(float32_bytes) / (1024**2), 3))


## 2. Dynamic-range quantization

Dynamic-range quantization typically quantizes model weights while keeping activations in floating point during inference. It is a useful first optimization because it does not require a representative dataset.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
dynamic_bytes = converter.convert()

dynamic_path = "../artifacts/cifar10_dynamic.tflite"
with open(dynamic_path, "wb") as f:
    f.write(dynamic_bytes)

print("Saved:", dynamic_path)
print("Dynamic model size (MiB):", round(len(dynamic_bytes) / (1024**2), 3))


## 3. Full integer INT8 quantization

The representative dataset shows the converter the range of inputs expected at inference. It must use the same preprocessing convention as the model's training pipeline: here, float32 RGB pixels scaled to 0–1.

We use a small subset to keep conversion practical.


In [ ]:
def representative_dataset():
    for image in x_test[:200:4]:
        yield [image[np.newaxis, ...].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

int8_bytes = converter.convert()
int8_path = "../artifacts/cifar10_int8.tflite"

with open(int8_path, "wb") as f:
    f.write(int8_bytes)

print("Saved:", int8_path)
print("INT8 model size (MiB):", round(len(int8_bytes) / (1024**2), 3))


## 4. Compare model sizes

Smaller files are easier to store and transfer, but file size alone does not establish that a model is better. We also need to measure accuracy and latency.


In [ ]:
sizes = {
    "Float32": len(float32_bytes),
    "Dynamic range": len(dynamic_bytes),
    "Full INT8": len(int8_bytes),
}

for name, size in sizes.items():
    print(f"{name:15s}: {size / (1024**2):.3f} MiB")

baseline = sizes["Float32"]
for name, size in sizes.items():
    print(f"{name:15s} reduction vs float32: {(1 - size / baseline) * 100:.1f}%")


## Reflection

1. Why is a representative dataset needed for full integer quantization?
2. Why must preprocessing remain consistent?
3. What trade-offs would matter on a Raspberry Pi?
4. Why should we benchmark accuracy and latency in addition to file size?

If full INT8 conversion fails due to an unsupported operation, keep the float32 and dynamic-range models as valid outputs and record the failure.
